# 🏁 Разбор шахматного анализатора (пошагово)

Привет! Этот ноутбук — тот же проект, но разбитый на маленькие шаги, чтобы можно было **запускать каждую ячейку отдельно** и смотреть, что реально возвращает библиотека `python-chess`.

Идея простая: сначала знакомимся с базовыми объектами (`Board`, `Move`, движок, книга), потом собираем из них функции анализатора, и в конце запускаем всё вместе на реальной партии.

**Что понадобится:**

- `python-chess` — библиотека (`pip install python-chess`)
- **Stockfish** — сам движок, скачивается отдельно (https://stockfishchess.org)
- Файл `Book.bin` — дебютная книга в polyglot формате
- TSV со списком дебютов (индексируется по EPD)
- PGN-файл с партией для анализа

Пути к файлам собраны в отдельной ячейке ниже — поменяй их под свою систему один раз.

## 0. Установка библиотеки

Если ещё не стоит:

In [ ]:
# Раскомментируй и запусти один раз
# !pip install python-chess

## 0.1. Настройка путей

Меняй значения под свою систему. Все остальные ячейки берут пути отсюда.

In [ ]:
BOOK_PATH     = "../material/Book.bin"                                  # polyglot книга (.bin)
OPENINGS_PATH = "../material/all.tsv"                                   # таблица названий дебютов (TSV)
PGN_PATH      = "../material/konteyni_vs_Snowstormme_2026.06.19.pgn"    # партия для анализа
STOCKFISH_PATH = "/opt/homebrew/Cellar/stockfish/18/bin/stockfish"      # исполняемый файл движка

# Windows пример:
# STOCKFISH_PATH = r"C:\stockfish\stockfish.exe"

## 1. Импорты

`python-chess` разбита на подмодули по задачам:

| Модуль | За что отвечает |
|--------|-----------------|
| `chess`          | Ядро: `Board`, `Move`, легальность ходов |
| `chess.pgn`      | Чтение и запись PGN-файлов |
| `chess.engine`   | Общение с UCI-движками (Stockfish и т.п.) |
| `chess.polyglot` | Работа с бинарными дебютными книгами |

In [ ]:
import chess
import chess.pgn
import chess.engine
import chess.polyglot
import math
import json
import csv

## 2. Знакомимся с `chess.Board`

Центральный объект всей библиотеки. Хранит позицию и умеет её менять.

In [ ]:
board = chess.Board()   # начальная позиция
print(board)            # красивая ASCII-доска
print()
print("Чей ход?", board.turn)          # True = белые, False = чёрные
print("Номер хода:", board.fullmove_number)

### 2.1. Словарь turns

`board.turn` — булев. Чтобы в отчёте был человекочитаемый `"white"` / `"black"`, используем словарь-переводчик.

In [ ]:
turns = {
    True: "white",
    False: "black",
}

print(turns[board.turn])   # "white"

## 3. FEN и EPD — как позиция превращается в строку

- **FEN** — полное описание позиции: расстановка фигур + чей ход + права на рокировку + взятие на проходе + счётчик полуходов + номер хода.
- **EPD** — то же самое, но **без счётчиков**. Именно поэтому EPD удобен как **идентификатор позиции**: две одинаковые позиции, пришедшие разными путями, дадут одинаковый EPD (а FEN может отличаться из-за счётчиков).

В коде: `board.fen()` и `board.epd()`.

In [ ]:
board = chess.Board()
print("FEN:", board.fen())
print("EPD:", board.epd())

Загрузить любую позицию можно, передав FEN в конструктор:

In [ ]:
# Позиция после 1.e4
b2 = chess.Board("rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 1")
print(b2)

## 4. Ходы: `Move`, UCI-нотация, push/pop

Ход — это объект `chess.Move`. Его текстовое представление — **UCI-нотация**: `клетка_откуда + клетка_куда` (+ буква фигуры при превращении).

Примеры: `e2e4`, `g1f3`, `e7e8q` (пешка e7 → e8 с превращением в ферзя).

In [ ]:
board = chess.Board()

# Создаём ход из UCI-строки
move = chess.Move.from_uci("e2e4")
print("Ход:", move.uci())

# push(...) — применить ход к доске
board.push(move)
print(board)
print()
print("Теперь ходят:", turns[board.turn])

### 4.1. push / pop как стек

`board.push(move)` кладёт ход в историю. `board.pop()` снимает последний ход и **возвращает** его. Это стек — очень удобно, когда надо "заглянуть в прошлое" и вернуться назад.

Мы используем этот трюк в `is_forced_move` и `is_book_move`.

In [ ]:
board = chess.Board()
board.push(chess.Move.from_uci("e2e4"))
board.push(chess.Move.from_uci("e7e5"))

print("После 1.e4 e5:")
print(board)
print()

last = board.pop()      # снимаем e7e5
print("Откатили:", last.uci())
print()
print(board)            # вернулись к состоянию после 1.e4

## 5. Легальные ходы

`board.legal_moves` — генератор всех легальных ходов из текущей позиции. Библиотека сама следит за шахами, связками, рокировкой, взятием на проходе.

Используем `.count()`, чтобы понять, был ли ход **вынужденным** (единственный легальный вариант).

In [ ]:
board = chess.Board()
print("Легальных ходов из старта:", board.legal_moves.count())

# Пример: детский мат в один. Позиция, где у чёрных единственный ответ
mate_in_one = chess.Board("rnbqkbnr/pppp1ppp/8/4p3/6P1/5P2/PPPPP2P/RNBQKBNR b KQkq - 0 2")
print("В этой позиции легальных ходов:", mate_in_one.legal_moves.count())

## 6. Дебютная книга (polyglot `.bin`)

`chess.polyglot.open_reader(path)` открывает бинарную книгу. Она содержит записи вида "хеш позиции → возможные ходы + веса".

Главный метод: `books.find_all(board)` — вернёт все ходы, записанные в книге для данной позиции.

In [ ]:
books = chess.polyglot.open_reader(BOOK_PATH)

# Что книга знает про стартовую позицию?
start = chess.Board()
for entry in books.find_all(start):
    print(f"{entry.move.uci()}  вес={entry.weight}")

`entry.move` — это объект `chess.Move`, его **можно сравнивать через `==`** с любым другим ходом. Именно так мы проверяем: "был ли сыгранный ход одним из книжных?".

## 7. Названия дебютов из TSV

Polyglot-книга знает *ходы*, но не *названия*. Названия загружаем из TSV, где ключ — EPD позиции.

Формат строк: `epd \t eco \t name`.

In [ ]:
def load_opening_book(path):
    opening_book = {}
    with open(path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            opening_book[row['epd']] = (row['eco'], row['name'])
    return opening_book

openings = load_opening_book(OPENINGS_PATH)
print("Всего дебютов:", len(openings))

# Пример: посмотрим, знает ли таблица стартовую позицию
print(openings.get(chess.Board().epd()))

## 8. Чтение партии из PGN

`chess.pgn.read_game(file)` разбирает файл и возвращает объект `Game` — дерево партии со всеми заголовками, ходами, вариантами и комментариями.

- `game.headers` — метаданные (`Event`, `White`, `Black`, `Result`, …).
- `game.board()` — стартовая позиция партии.
- `game.mainline_moves()` — итератор по основной линии (без побочных вариантов).

In [ ]:
pgn = open(PGN_PATH)
game = chess.pgn.read_game(pgn)

print("Белые:", game.headers.get("White"))
print("Чёрные:", game.headers.get("Black"))
print("Результат:", game.headers.get("Result"))

board = game.board()
print("\nСтартовая позиция:")
print(board)

# Первые 5 ходов партии
moves = list(game.mainline_moves())
print("\nПервые 5 ходов:", [m.uci() for m in moves[:5]])

## 9. Запуск Stockfish

`chess.engine.SimpleEngine.popen_uci(path)` **запускает движок как отдельный процесс** и открывает канал общения по UCI-протоколу. Все команды идут через объект `engine`.

⚠️ Обязательно потом вызвать `engine.quit()`, иначе процесс останется висеть.

In [ ]:
engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
print("Движок запущен:", engine.id.get("name"))

## 10. Анализ позиции: `engine.analyse`

Просим движок подумать над позицией. Ограничение можно задать по глубине, времени или узлам:

```python
chess.engine.Limit(depth=16)        # думать до глубины 16
chess.engine.Limit(time=1.0)        # думать 1 секунду
chess.engine.Limit(nodes=100_000)   # ограничить количеством узлов
```

Возвращает словарь `info` с ключами:
- `info["score"]` — оценка (объект `PovScore`).
- `info["pv"]` — **Principal Variation**, список ходов лучшей линии. `pv[0]` — лучший ход.
- `info["depth"]`, `info["nodes"]`, `info["time"]` — метаданные.

In [ ]:
board = chess.Board()
info = engine.analyse(board, chess.engine.Limit(depth=16))

print("Ключи info:", list(info.keys()))
print()
print("Оценка:", info["score"])
print("Лучший ход:", info["pv"][0].uci())
print("PV:", [m.uci() for m in info["pv"][:5]])

### 10.1. PovScore, `.white()`, обработка мата

`PovScore` хранит оценку с точки зрения того, чей ход. Чтобы всегда сравнивать в одной системе координат, приводим к белым: `info["score"].white()`.

Метод `.score(mate_score=10000)` вытаскивает число в сантипешках. Если у движка форсированный мат — обычный `.score()` вернёт `None`, поэтому мы указываем `mate_score`: белые матуют → `+10000`, белым ставят мат → `-10000`.

`.is_mate()` — булев флаг, есть ли форсированный мат вообще.
`.mate()` — через сколько ходов мат (плюс/минус).

In [ ]:
eval_cp = info["score"].white().score(mate_score=10000)
print("Оценка в сантипешках (у белых):", eval_cp)
print("Это мат?", info["score"].is_mate())

# Пример с матовой позицией: белые матуют в 1
mate_pos = chess.Board("6k1/5ppp/8/8/8/8/5PPP/R5K1 w - - 0 1")
info_mate = engine.analyse(mate_pos, chess.engine.Limit(depth=8))
print("\nВ матовой позиции:")
print("  Мат?", info_mate["score"].is_mate())
print("  Мат в:", info_mate["score"].white().mate(), "ход(ов)")
print("  Оценка с mate_score:", info_mate["score"].white().score(mate_score=10000))

## 11. Функция `calculate_winrate`

Сигмоида, которая переводит оценку в сантипешках в **процент вероятности выигрыша**.

Зачем это нужно: напрямую сравнивать сантипешки нельзя. Разница между `+100` и `+200` намного важнее, чем между `+800` и `+900` (там и так уже выиграно). Сигмоида это нормализует.

Коэффициент `0.00368208` — эмпирическая константа (та же, что у Lichess).

In [ ]:
def calculate_winrate(x):
    y = 50 + 50 * (2 / (1 + math.exp(-0.00368208 * x)) - 1)
    return y

# Проверим на нескольких значениях
for cp in [-1000, -300, -100, 0, 100, 300, 1000]:
    print(f"eval={cp:+5d} cp  →  winrate={calculate_winrate(cp):.1f}%")

## 12. Классификация обычных ходов

Если ход не совпал с лучшим — смотрим, на сколько процентов винрейта просела позиция.

| Потери винрейта | Категория |
|---|---|
| ≤ 2% | Excellent |
| ≤ 5% | Good |
| ≤ 10% | Inaccuracy |
| ≤ 20% | Mistake |
| > 20% | Blunder |

In [ ]:
def classic_moves(move, best_moves, winrate_change):
    if move == best_moves[0]:
        return "Best"
    elif winrate_change <= 2:
        return "Excellent"
    elif winrate_change <= 5:
        return "Good"
    elif winrate_change <= 10:
        return "Inaccuracy"
    elif winrate_change <= 20:
        return "Mistake"
    else:
        return "Blunder"

## 13. `is_forced_move` — был ли ход единственным?

Хитрость: функцию вызывают **уже после** `push(move)`, то есть на доске новая позиция. Чтобы узнать, сколько ходов было доступно **до** хода, делаем pop → считаем → push обратно.

Так доска остаётся в том же состоянии, в котором была до вызова функции. Это ключевое свойство — иначе внешний цикл сломается.

In [ ]:
def is_forced_move(board):
    last_move = board.pop()
    forced = board.legal_moves.count() == 1
    board.push(last_move)
    return forced

## 14. `is_book_move` — был ли ход из теории?

Тот же приём с pop/push. Логика:

1. Откатываем последний ход.
2. Спрашиваем книгу: какие ходы она знает для этой позиции? Собираем список.
3. Накатываем ход обратно.
4. По EPD **новой** (после-ходовой) позиции ищем название дебюта в TSV. Если нашли — записываем в отчёт.
5. Проверяем: был ли реально сыгранный ход в списке книжных.

📌 EPD берётся именно от позиции **после** хода — так же, как ключи проиндексированы в TSV.

In [ ]:
def is_book_move(board, report):
    last_move = board.pop()

    continuations = []
    for continuation in books.find_all(board):
        continuations.append(continuation.move)
    board.push(last_move)

    opening = openings.get(board.epd())
    if opening is not None:
        report['opening'] = opening[1]

    return last_move in continuations

## 15. `categorize_move` — общая логика классификации

Приоритет проверок:

1. **Forced** — всегда первым. Если выбора не было, категоризировать бессмысленно.
2. **Book** — если позиция в книге **и** дебют ещё не закончился.
3. Как только встретился хоть один не-книжный ход — навсегда помечаем `openings_ended = True`. Дальше даже если случайно совпадёт с книгой — уже миттельшпиль.
4. Иначе — считаем изменение винрейта и передаём в `classic_moves`.

📌 Про знак `winrate_change`: для **белых** ухудшение = снижение оценки → `prev - current`. Для **чёрных** ухудшение = **рост** оценки белых → `current - prev`. В обоих случаях результат — потери в процентах для того, кто ходил.

📌 `openings_state` — словарь, а не переменная. Это чтобы **изменения внутри функции были видны снаружи** (словари в Python передаются по ссылке).

In [ ]:
def categorize_move(eval, prev_eval, mover, move, best_moves, board, openings_state, report):
    if is_forced_move(board):
        return "Forced"

    if is_book_move(board, report) and not openings_state['openings_ended']:
        return "Book"
    else:
        openings_state['openings_ended'] = True

    current_winrate = calculate_winrate(eval)
    prev_winrate = calculate_winrate(prev_eval)
    if mover == "white":
        winrate_change = prev_winrate - current_winrate
    else:
        winrate_change = current_winrate - prev_winrate
    return classic_moves(move, best_moves, winrate_change)

## 16. `analyse_game` — главный цикл

Собираем всё вместе. По шагам на каждой итерации:

1. **До хода** запоминаем `mover`, `move_number`, `best_moves` (из PV предыдущего `info`).
2. `board.push(move)` — делаем ход.
3. **После хода** снимаем `to_move` и `board.fen()`.
4. Анализируем новую позицию → получаем новую оценку.
5. Классифицируем через `categorize_move`.
6. Если позиция матовая — записываем `mate_in`.
7. Сохраняем в `report` и обновляем `prev_eval`.

Один тонкий момент: `best_moves = info["pv"]` берётся **до** нового `analyse` — то есть это PV из позиции **перед** ходом. Именно с ним и надо сравнивать сыгранный ход.

In [ ]:
def analyse_game(pgn):
    report = {
        "game_id": 'abc123',
        "opening": '',
        "moves": [],
        "eval_history": [],
    }

    openings_state = {"openings_ended": False}

    game = chess.pgn.read_game(pgn)
    board = game.board()

    info = engine.analyse(board, chess.engine.Limit(depth=16))
    prev_eval = info["score"].white().score(mate_score=10000)

    for move in game.mainline_moves():
        mover = turns[board.turn]
        move_number = board.fullmove_number
        best_moves = info["pv"]

        board.push(move)
        to_move = turns[board.turn]

        move_info = {
            "move_number": move_number,
            "move": move.uci(),
            "mover": mover,
            "to_move": to_move,
            "mate_in": None,
            "evaluation": None,
            "category": None,
            "fen": board.fen(),
        }

        info = engine.analyse(board, chess.engine.Limit(depth=16))
        eval = info["score"].white().score(mate_score=10000)

        category = categorize_move(
            eval, prev_eval, mover, move, best_moves,
            board, openings_state, report,
        )
        move_info["category"] = category

        print(f"{move_number}. {mover} {move.uci()}  "
              f"{prev_eval} → {eval}  [{category}]")

        prev_eval = eval

        if info["score"].is_mate():
            move_info["mate_in"] = info["score"].white().mate()

        move_info["evaluation"] = eval
        report["moves"].append(move_info)
        report["eval_history"].append(eval)

    return report

## 17. Запускаем анализ на реальной партии

In [ ]:
pgn = open(PGN_PATH)
report = analyse_game(pgn)

# Красивый JSON-вывод
print(json.dumps(report, indent=2, ensure_ascii=False))

## 18. Закрываем движок

⚠️ Обязательно — иначе процесс Stockfish останется висеть в фоне.

In [ ]:
engine.quit()
print("Движок закрыт.")

## 🎯 Что дальше?

Точки для расширения:

- **Новые категории** (Great, Brilliant, Miss) — в `classic_moves` / `categorize_move`.
- **Метрики позиции** (активность фигур, безопасность короля) — внутри цикла `analyse_game`, там где формируется `move_info`.
- **Несколько партий из одного PGN** — `chess.pgn.read_game` можно вызывать в цикле, пока не вернёт `None`.
- **График винрейта** — `report["eval_history"]` уже готов, просто подать в matplotlib.

Если по какой-то ячейке будут вопросы — пиши, разберёмся.